In [1]:
#导入包和路径
from pathlib import Path
from datetime import datetime, timezone
import re
import traceback

import numpy as np
import pandas as pd
import h5py
import remfile

from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO
from tqdm import tqdm

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RESULTS_DIR = PROJECT_ROOT / "results"
CACHE_DIR = PROJECT_ROOT / "cache" / "remfile"
OUTPUT_DIR = RESULTS_DIR / "global_inventory"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Cache:", CACHE_DIR)
print("Output:", OUTPUT_DIR)

Project root: D:\Ephys_analysis\ibl-population-dynamics
Cache: D:\Ephys_analysis\ibl-population-dynamics\cache\remfile
Output: D:\Ephys_analysis\ibl-population-dynamics\results\global_inventory


In [2]:
#参数
DANDISET_ID = "000409"
VERSION = "0.260309.1324"

GOOD_UNIT_SCORE = 1.0
MIN_RT_S = 0.08
MAX_RT_S = 2.00

# 第一次先测试 3 个；确认无误后改成 None
MAX_SESSIONS = None
CHECKPOINT_EVERY = 5

SESSION_SUMMARY_PATH = OUTPUT_DIR / "global_session_summary.csv"
REGION_INVENTORY_PATH = OUTPUT_DIR / "global_region_inventory.csv"
TRIAL_INVENTORY_PATH = OUTPUT_DIR / "global_trial_condition_inventory.csv"
ERROR_LOG_PATH = OUTPUT_DIR / "global_inventory_errors.csv"

In [3]:
#读取inventory
inventory_path = RESULTS_DIR / "session_inventory.csv"
session_inventory = pd.read_csv(
    inventory_path,
    dtype={"subject": "string", "session": "string", "asset_path": "string"},
)

required = {"subject", "session", "asset_path"}
missing = required - set(session_inventory.columns)
if missing:
    raise ValueError(f"Missing inventory columns: {sorted(missing)}")

print("Rows:", len(session_inventory))
print("Subjects:", session_inventory["subject"].nunique())
print("Sessions:", session_inventory["session"].nunique())
display(session_inventory.head())

Rows: 459
Subjects: 139
Sessions: 459


,subject,session,asset_path
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...
1,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...
2,CSH-ZAD-011,5b44c40f-80f4-44fb-abfb-c7f19e27a6ca,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-5b44c40f-8...
3,CSH-ZAD-011,da188f2c-553c-4e04-879b-c9ea2d1b9a93,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-da188f2c-5...
4,CSH-ZAD-019,49e0ab27-827a-4c91-bcaa-97eea27a1b8d,sub-CSH-ZAD-019/sub-CSH-ZAD-019_ses-49e0ab27-8...


In [4]:
#辅助函数
def scalar_to_text(value):
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if pd.isna(value):
        return pd.NA
    return str(value)



def array_to_text(values):
    return pd.Series(
        [scalar_to_text(value) for value in values],
        dtype="string",
    )

def collapse_cortical_layer(region_name):
    if pd.isna(region_name):
        return pd.NA
    region_name = str(region_name).strip()
    pattern = r",?\s+layer\s+(1|2/3|2|3|4|5|6a|6b|6)$"
    return re.sub(pattern, "", region_name, flags=re.IGNORECASE).strip()



In [5]:
#引用max_electrode
def row_from_reference(reference, electrodes_df):
    if isinstance(reference, pd.Series):
        return reference
    if isinstance(reference, pd.DataFrame):
        return None if reference.empty else reference.iloc[0]

    if isinstance(reference, (list, tuple, np.ndarray)):
        values = np.asarray(reference).ravel()
        if len(values) and np.issubdtype(values.dtype, np.integer):
            idx = int(values[0])
            if 0 <= idx < len(electrodes_df):
                return electrodes_df.iloc[idx]
        return None

    if isinstance(reference, (int, np.integer)):
        idx = int(reference)
        if idx in electrodes_df.index:
            row = electrodes_df.loc[idx]
            return row.iloc[0] if isinstance(row, pd.DataFrame) else row
        if 0 <= idx < len(electrodes_df):
            return electrodes_df.iloc[idx]
    return None


def get_representative_electrode(nwbfile, electrodes_df, unit_index):
    max_ref = nwbfile.units["max_electrode"][unit_index]
    row = row_from_reference(max_ref, electrodes_df)
    if row is not None:
        return row

    linked = nwbfile.units["electrodes"][unit_index]
    if isinstance(linked, pd.Series):
        linked_df = linked.to_frame().T
    elif isinstance(linked, pd.DataFrame):
        linked_df = linked
    elif isinstance(linked, (list, tuple, np.ndarray)):
        values = np.asarray(linked).ravel()
        if len(values) and np.issubdtype(values.dtype, np.integer):
            values = values.astype(int)
            values = values[(values >= 0) & (values < len(electrodes_df))]
            linked_df = electrodes_df.iloc[values]
        else:
            linked_df = pd.DataFrame()
    else:
        linked_df = pd.DataFrame()

    if linked_df.empty or "location" not in linked_df.columns:
        return None

    locations = linked_df["location"].dropna().astype(str).unique()
    return linked_df.iloc[0] if len(locations) == 1 else None

In [6]:
#汇总session的units和regions
def read_unit_column(nwbfile, name, default=np.nan):
    n_units = len(nwbfile.units)

    if name not in nwbfile.units.colnames:
        return np.full(
            n_units,
            default,
            dtype=object,
        )

    return np.asarray(
        nwbfile.units[name][:]
    )


def summarize_units_by_region(
    nwbfile,
    subject,
    session,
    asset_path,
):
    n_units = len(nwbfile.units)

    quality = pd.to_numeric(
        pd.Series(
            read_unit_column(
                nwbfile,
                "ibl_quality_score",
            )
        ),
        errors="coerce",
    )

    firing_rate = pd.to_numeric(
        pd.Series(
            read_unit_column(
                nwbfile,
                "firing_rate",
            )
        ),
        errors="coerce",
    )

    probe_name = array_to_text(
        read_unit_column(
            nwbfile,
            "probe_name",
            pd.NA,
        )
    )

    electrodes_df = nwbfile.electrodes.to_dataframe()
    region_raw = []

    for unit_index in range(n_units):
        electrode = get_representative_electrode(
            nwbfile,
            electrodes_df,
            unit_index,
        )

        if electrode is None:
            region_raw.append(pd.NA)
        else:
            region_raw.append(
                scalar_to_text(
                    electrode.get(
                        "location",
                        pd.NA,
                    )
                )
            )

    units_light = pd.DataFrame(
        {
            "unit_index": np.arange(n_units),
            "probe_name": probe_name,
            "ibl_quality_score": quality,
            "firing_rate": firing_rate,
            "region_raw": pd.Series(
                region_raw,
                dtype="string",
            ),
        }
    )

    units_light["region_parent"] = (
        units_light["region_raw"]
        .apply(collapse_cortical_layer)
        .astype("string")
    )

    units_light["is_good_unit"] = (
        units_light["ibl_quality_score"]
        .eq(GOOD_UNIT_SCORE)
    )

    # 每行保留一个原始脑区名称，包括 layer 信息。
    region_summary = (
        units_light
        .dropna(subset=["region_raw"])
        .groupby(
            [
                "region_raw",
                "region_parent",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            n_all_units=("unit_index", "size"),
            n_good_units=("is_good_unit", "sum"),
            n_probes=("probe_name", "nunique"),
            median_firing_rate_hz=(
                "firing_rate",
                "median",
            ),
        )
    )

    region_summary.insert(
        0,
        "asset_path",
        str(asset_path),
    )
    region_summary.insert(
        0,
        "session",
        str(session),
    )
    region_summary.insert(
        0,
        "subject",
        str(subject),
    )

    probes = sorted(
        probe_name.dropna().astype(str).unique()
    )

    session_unit_summary = {
        "subject": str(subject),
        "session": str(session),
        "asset_path": str(asset_path),
        "n_units": int(n_units),
        "n_good_units": int(
            units_light["is_good_unit"].sum()
        ),
        "n_units_with_region": int(
            units_light["region_raw"].notna().sum()
        ),
        "n_probes": int(
            probe_name.dropna().nunique()
        ),
        "probe_names": " | ".join(probes),
    }

    return session_unit_summary, region_summary

In [7]:
#汇总session的trails
REQUIRED_TRIAL_COLUMNS = [
    "gabor_stimulus_onset_time",
    "wheel_movement_onset_time",
    "choice_registration_time",
    "feedback_time",
    "mouse_wheel_choice",
    "probability_left",
    "is_mouse_rewarded",
]

TRIAL_CONDITION_COLUMNS = [
    "gabor_stimulus_contrast",
    "gabor_stimulus_side",
    "probability_left",
    "block_type",
    "mouse_wheel_choice",
    "is_mouse_rewarded",
]


def summarize_trials(nwbfile, subject, session, asset_path):
    if nwbfile.trials is None:
        raise ValueError("No trials table")

    trials = nwbfile.trials.to_dataframe().reset_index()
    if "id" in trials.columns:
        trials = trials.rename(columns={"id": "trial_index"})
    elif "index" in trials.columns:
        trials = trials.rename(columns={"index": "trial_index"})
    else:
        trials.insert(0, "trial_index", np.arange(len(trials)))

    missing = [c for c in REQUIRED_TRIAL_COLUMNS + TRIAL_CONDITION_COLUMNS if c not in trials.columns]
    if missing:
        raise ValueError(f"Missing trial columns: {sorted(set(missing))}")

    for c in ["gabor_stimulus_side", "block_type", "mouse_wheel_choice"]:
        trials[c] = trials[c].map(scalar_to_text)

    trials["reaction_time_s"] = (
        pd.to_numeric(trials["wheel_movement_onset_time"], errors="coerce")
        - pd.to_numeric(trials["gabor_stimulus_onset_time"], errors="coerce")
    )
    trials["has_required_events"] = trials[REQUIRED_TRIAL_COLUMNS].notna().all(axis=1)
    trials["rt_in_range"] = trials["reaction_time_s"].between(MIN_RT_S, MAX_RT_S, inclusive="both")
    trials["is_valid_trial"] = trials["has_required_events"] & trials["rt_in_range"]
    trials["valid_rt_s"] = np.where(trials["is_valid_trial"], trials["reaction_time_s"], np.nan)

    condition_summary = (
        trials.groupby(TRIAL_CONDITION_COLUMNS, dropna=False, as_index=False)
        .agg(
            n_trials=("trial_index", "size"),
            n_valid_trials=("is_valid_trial", "sum"),
            median_rt_valid_s=("valid_rt_s", "median"),
        )
    )
    condition_summary.insert(0, "asset_path", str(asset_path))
    condition_summary.insert(0, "session", str(session))
    condition_summary.insert(0, "subject", str(subject))

    valid = trials.loc[trials["is_valid_trial"]]
    reward_numeric = pd.to_numeric(valid["is_mouse_rewarded"], errors="coerce")
    session_trial_summary = {
        "n_trials": int(len(trials)),
        "n_valid_trials": int(trials["is_valid_trial"].sum()),
        "valid_trial_fraction": float(trials["is_valid_trial"].mean()),
        "n_missing_required_events": int((~trials["has_required_events"]).sum()),
        "n_rt_out_of_range": int((trials["has_required_events"] & ~trials["rt_in_range"]).sum()),
        "n_contrasts": int(valid["gabor_stimulus_contrast"].nunique(dropna=True)),
        "n_stimulus_sides": int(valid["gabor_stimulus_side"].nunique(dropna=True)),
        "n_probability_left_values": int(valid["probability_left"].nunique(dropna=True)),
        "n_block_types": int(valid["block_type"].nunique(dropna=True)),
        "median_rt_valid_s": float(valid["reaction_time_s"].median()),
        "accuracy_valid": float(reward_numeric.mean()),
    }
    return session_trial_summary, condition_summary

In [8]:
#访问NWB
def process_one_session(row, dandiset, cache_dir):
    subject = str(row["subject"])
    session = str(row["session"])
    asset_path = str(row["asset_path"])

    rem_file = h5py_file = io = None
    try:
        asset = dandiset.get_asset_by_path(path=asset_path)
        s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
        disk_cache = remfile.DiskCache(str(cache_dir))
        rem_file = remfile.File(s3_url, disk_cache=disk_cache)
        h5py_file = h5py.File(rem_file, mode="r")
        io = NWBHDF5IO(file=h5py_file, load_namespaces=True)
        nwbfile = io.read()

        unit_summary, region_summary = summarize_units_by_region(
            nwbfile, subject, session, asset_path
        )
        trial_summary, trial_condition_summary = summarize_trials(
            nwbfile, subject, session, asset_path
        )

        session_summary = {
            **unit_summary,
            **trial_summary,
            "dandiset_id": DANDISET_ID,
            "dandiset_version": VERSION,
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        return pd.DataFrame([session_summary]), region_summary, trial_condition_summary

    finally:
        for obj in (io, h5py_file, rem_file):
            if obj is not None:
                try:
                    obj.close()
                except Exception:
                    pass

In [9]:
#checkpoint
def load_csv_or_empty(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def save_tables(session_df, region_df, trial_df, errors_df):
    if len(session_df):
        session_df = session_df.drop_duplicates("session", keep="last").sort_values(["subject", "session"])
        session_df.to_csv(SESSION_SUMMARY_PATH, index=False)

    if len(region_df):
        region_df = (
            region_df
            .drop_duplicates(
                [
                    "session",
                    "region_raw",
                ],
                keep="last",
            )
            .sort_values(
                [
                    "subject",
                    "session",
                    "region_parent",
                    "region_raw",
                ]
            )
        )

        region_df.to_csv(
            REGION_INVENTORY_PATH,
            index=False,
        )

    if len(trial_df):
        keys = ["session", *TRIAL_CONDITION_COLUMNS]
        trial_df = trial_df.drop_duplicates(keys, keep="last").sort_values(
            ["subject", "session", "gabor_stimulus_contrast"]
        )
        trial_df.to_csv(TRIAL_INVENTORY_PATH, index=False)

    if len(errors_df):
        errors_df = errors_df.drop_duplicates("session", keep="last").sort_values(["subject", "session"])
        errors_df.to_csv(ERROR_LOG_PATH, index=False)

    return session_df.reset_index(drop=True), region_df.reset_index(drop=True), trial_df.reset_index(drop=True), errors_df.reset_index(drop=True)

In [14]:
#读取checkpoint并决定session对象
session_summary_df = load_csv_or_empty(SESSION_SUMMARY_PATH)
region_inventory_df = load_csv_or_empty(REGION_INVENTORY_PATH)
trial_inventory_df = load_csv_or_empty(TRIAL_INVENTORY_PATH)
errors_df = load_csv_or_empty(ERROR_LOG_PATH)

completed = set(session_summary_df["session"].astype(str)) if "session" in session_summary_df.columns else set()
sessions_to_process = session_inventory.loc[
    ~session_inventory["session"].astype(str).isin(completed)
].copy()

if MAX_SESSIONS is not None:
    sessions_to_process = sessions_to_process.head(MAX_SESSIONS).copy()

print("Already completed:", len(completed))
print("Selected for this run:", len(sessions_to_process))

Already completed: 453
Selected for this run: 6


In [15]:
#补充
if "session" in session_summary_df.columns:
    completed_sessions = set(
        session_summary_df["session"].astype(str)
    )
else:
    completed_sessions = set()

sessions_to_process = (
    session_inventory.loc[
        ~session_inventory["session"]
        .astype(str)
        .isin(completed_sessions)
    ]
    .copy()
)

if MAX_SESSIONS is not None:
    sessions_to_process = (
        sessions_to_process
        .head(MAX_SESSIONS)
        .copy()
    )

print("Already completed:", len(completed_sessions))
print("Sessions selected for this run:", len(sessions_to_process))

display(
    sessions_to_process[
        ["subject", "session", "asset_path"]
    ]
)

Already completed: 453
Sessions selected for this run: 6


,subject,session,asset_path
448,ibl-witten-27,f304211a-81b1-446f-a435-25e589fe3a5a,sub-ibl-witten-27/sub-ibl-witten-27_ses-f30421...
449,ibl-witten-29,0cc486c3-8c7b-494d-aa04-b70e2690bcba,sub-ibl-witten-29/sub-ibl-witten-29_ses-0cc486...
450,ibl-witten-29,58b1e920-cfc8-467e-b28b-7654a55d0977,sub-ibl-witten-29/sub-ibl-witten-29_ses-58b1e9...
451,ibl-witten-29,69a0e953-a643-4f0e-bb26-dc65af3ea7d7,sub-ibl-witten-29/sub-ibl-witten-29_ses-69a0e9...
452,ibl-witten-29,72028382-a869-4745-bacf-cb8789e16953,sub-ibl-witten-29/sub-ibl-witten-29_ses-720283...
453,ibl-witten-29,821f1883-27f3-411d-afd3-fb8241bbc39a,sub-ibl-witten-29/sub-ibl-witten-29_ses-821f18...


In [16]:
#loop
with DandiAPIClient() as client:
    dandiset = client.get_dandiset(dandiset_id=DANDISET_ID, version_id=VERSION)

    for run_index, (_, row) in enumerate(
        tqdm(sessions_to_process.iterrows(), total=len(sessions_to_process), desc="Scanning NWB sessions"),
        start=1,
    ):
        try:
            one_session, one_region, one_trial = process_one_session(row, dandiset, CACHE_DIR)
            session_summary_df = pd.concat([session_summary_df, one_session], ignore_index=True)
            region_inventory_df = pd.concat([region_inventory_df, one_region], ignore_index=True)
            trial_inventory_df = pd.concat([trial_inventory_df, one_trial], ignore_index=True)

            if len(errors_df) and "session" in errors_df.columns:
                errors_df = errors_df.loc[errors_df["session"].astype(str) != str(row["session"])].copy()

        except Exception as exc:
            error_row = pd.DataFrame([{
                "subject": str(row["subject"]),
                "session": str(row["session"]),
                "asset_path": str(row["asset_path"]),
                "error_type": type(exc).__name__,
                "error_message": str(exc),
                "traceback": traceback.format_exc(),
                "failed_at_utc": datetime.now(timezone.utc).isoformat(),
            }])
            errors_df = pd.concat([errors_df, error_row], ignore_index=True)
            print("\nFailed:", row["session"], type(exc).__name__, str(exc))

        if run_index % CHECKPOINT_EVERY == 0 or run_index == len(sessions_to_process):
            session_summary_df, region_inventory_df, trial_inventory_df, errors_df = save_tables(
                session_summary_df, region_inventory_df, trial_inventory_df, errors_df
            )
            print(f"\nCheckpoint saved after {run_index} session(s) in this run.")

Scanning NWB sessions:  83%|████████▎ | 5/6 [50:52<12:28, 748.56s/it]


Checkpoint saved after 5 session(s) in this run.


Scanning NWB sessions: 100%|██████████| 6/6 [1:07:13<00:00, 672.20s/it]


Checkpoint saved after 6 session(s) in this run.


In [17]:
print("Session summary:", session_summary_df.shape)
print("Region inventory:", region_inventory_df.shape)
print("Trial inventory:", trial_inventory_df.shape)
print("Errors:", errors_df.shape)

display(session_summary_df.head())
display(region_inventory_df.head())
display(trial_inventory_df.head())

if len(errors_df):
    display(errors_df[["subject", "session", "error_type", "error_message"]].head(20))

Session summary: (459, 22)
Region inventory: (8231, 9)
Trial inventory: (24472, 12)
Errors: (3, 7)


,subject,session,asset_path,n_units,n_good_units,n_units_with_region,n_probes,probe_names,n_trials,n_valid_trials,...,n_rt_out_of_range,n_contrasts,n_stimulus_sides,n_probability_left_values,n_block_types,median_rt_valid_s,accuracy_valid,dandiset_id,dandiset_version,processed_at_utc
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,519,50,519,1,Probe00,854,406,...,390,5,2,3,3,0.138805,0.894089,409,0.260309.1324,2026-07-25T23:19:56.748497+00:00
1,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...,1341,119,1341,2,Probe00 | Probe01,415,307,...,106,5,2,3,3,0.160009,0.798046,409,0.260309.1324,2026-07-25T23:20:00.950188+00:00
2,CSH-ZAD-011,5b44c40f-80f4-44fb-abfb-c7f19e27a6ca,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-5b44c40f-8...,1927,134,1927,2,Probe00 | Probe01,591,437,...,150,5,2,3,3,0.183382,0.926773,409,0.260309.1324,2026-07-25T23:20:06.143089+00:00
3,CSH-ZAD-011,da188f2c-553c-4e04-879b-c9ea2d1b9a93,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-da188f2c-5...,2019,205,2019,2,Probe00 | Probe01,927,676,...,250,5,2,3,3,0.152621,0.933432,409,0.260309.1324,2026-07-26T01:27:48.527294+00:00
4,CSH-ZAD-019,49e0ab27-827a-4c91-bcaa-97eea27a1b8d,sub-CSH-ZAD-019/sub-CSH-ZAD-019_ses-49e0ab27-8...,1689,141,1689,2,Probe00 | Probe01,884,576,...,305,5,2,3,3,0.187889,0.878472,409,0.260309.1324,2026-07-26T01:27:53.792903+00:00


,subject,session,asset_path,region_raw,region_parent,n_all_units,n_good_units,n_probes,median_firing_rate_hz
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,Fields of Forel,Fields of Forel,37,6,1,8.037428
1,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,Midbrain,Midbrain,225,18,1,8.460120
2,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,Midbrain reticular nucleus,Midbrain reticular nucleus,47,4,1,11.107022
3,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,Precommissural nucleus,Precommissural nucleus,53,3,1,11.380019
4,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,Subparafascicular nucleus parvicellular part,Subparafascicular nucleus parvicellular part,20,1,1,11.030513


,subject,session,asset_path,gabor_stimulus_contrast,gabor_stimulus_side,probability_left,block_type,mouse_wheel_choice,is_mouse_rewarded,n_trials,n_valid_trials,median_rt_valid_s
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,0.0,left,0.2,right_block,clockwise,True,3,1,0.100523
1,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,0.0,left,0.2,right_block,counter_clockwise,False,4,3,0.133649
2,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,0.0,left,0.5,unbiased,clockwise,True,2,2,0.723654
3,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,0.0,left,0.5,unbiased,counter_clockwise,False,3,1,0.145045
4,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...,0.0,left,0.8,left_block,clockwise,True,35,9,0.119305


,subject,session,error_type,error_message
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,NameError,name 'get_region_acronym' is not defined
1,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,NameError,name 'get_region_acronym' is not defined
2,CSH-ZAD-011,5b44c40f-80f4-44fb-abfb-c7f19e27a6ca,NameError,name 'get_region_acronym' is not defined


In [18]:
parent_region_session_inventory = (
    region_inventory_df
    .groupby(
        [
            "subject",
            "session",
            "asset_path",
            "region_parent",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        n_all_units=("n_all_units", "sum"),
        n_good_units=("n_good_units", "sum"),
        n_probes=("n_probes", "max"),
    )
)

region_dataset_overview = (
    parent_region_session_inventory
    .groupby(
        "region_parent",
        as_index=False,
    )
    .agg(
        n_sessions=("session", "nunique"),
        n_subjects=("subject", "nunique"),
        total_good_units=("n_good_units", "sum"),
        median_good_units_per_session=(
            "n_good_units",
            "median",
        ),
        max_good_units_in_session=(
            "n_good_units",
            "max",
        ),
        n_sessions_ge_10=(
            "n_good_units",
            lambda values: values.ge(10).sum(),
        ),
        n_sessions_ge_20=(
            "n_good_units",
            lambda values: values.ge(20).sum(),
        ),
    )
    .sort_values(
        [
            "n_sessions_ge_20",
            "n_sessions_ge_10",
            "total_good_units",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(region_dataset_overview.head(50))

,region_parent,n_sessions,n_subjects,total_good_units,median_good_units_per_session,max_good_units_in_session,n_sessions_ge_10,n_sessions_ge_20
0,Midbrain reticular nucleus,117,74,2678,19.0,91,80,58
1,Caudoputamen,78,43,2892,27.5,136,56,44
2,Lateral posterior nucleus of the thalamus,79,66,1812,18.0,91,51,38
3,Posterior complex of the thalamus,51,44,2329,35.0,150,40,35
4,Anterior pretectal nucleus,53,44,1543,25.0,74,42,33
5,Field CA1,142,96,1807,8.5,131,65,27
6,Anterior area,49,44,940,20.0,72,30,25
7,Ventral posteromedial nucleus of the thalamus,39,32,1310,30.0,117,29,24
8,Primary motor area,34,23,1458,26.0,214,28,24
9,arbor vitae,89,45,1684,7.0,153,39,23


In [19]:
#查询
MIN_GOOD_UNITS_SCREEN = 10

visp_candidates = (
    parent_region_session_inventory.loc[
        parent_region_session_inventory[
            "region_parent"
        ].eq("Primary visual area")
        & parent_region_session_inventory[
            "n_good_units"
        ].ge(MIN_GOOD_UNITS_SCREEN)
    ]
    .sort_values(
        "n_good_units",
        ascending=False,
    )
)

mos_candidates = (
    parent_region_session_inventory.loc[
        parent_region_session_inventory[
            "region_parent"
        ].eq("Secondary motor area")
        & parent_region_session_inventory[
            "n_good_units"
        ].ge(MIN_GOOD_UNITS_SCREEN)
    ]
    .sort_values(
        "n_good_units",
        ascending=False,
    )
)

print("VISp candidates:", len(visp_candidates))
display(visp_candidates.head(30))

print("MOs candidates:", len(mos_candidates))
display(mos_candidates.head(30))

VISp candidates: 28


,subject,session,asset_path,region_parent,n_all_units,n_good_units,n_probes
173,CSH-ZAD-022,41431f53-69fd-4e3b-80ce-ea62e03bf9c7,sub-CSH-ZAD-022/sub-CSH-ZAD-022_ses-41431f53-6...,Primary visual area,829,63,1
352,CSH-ZAD-025,6274dda8-3a59-4aa1-95f8-a8a549c46a26,sub-CSH-ZAD-025/sub-CSH-ZAD-025_ses-6274dda8-3...,Primary visual area,300,62,1
5810,ZFM-01577,09b2c4d1-058d-4c84-9fd4-97530f85baf6,sub-ZFM-01577/sub-ZFM-01577_ses-09b2c4d1-058d-...,Primary visual area,264,51,1
938,CSHL052,85dc2ebd-8aaf-46b0-9284-a197aee8b16f,sub-CSHL052/sub-CSHL052_ses-85dc2ebd-8aaf-46b0...,Primary visual area,222,49,1
4617,PL050,2584ce3c-db10-4076-89cb-5d313138dd38,sub-PL050/sub-PL050_ses-2584ce3c-db10-4076-89c...,Primary visual area,296,44,1
3016,NR-0021,e0928e11-2b86-4387-a203-80c77fab5d52,sub-NR-0021/sub-NR-0021_ses-e0928e11-2b86-4387...,Primary visual area,276,40,2
2727,MFD-05,ca4ecb4c-4b60-4723-9b9e-2c54a6290a53,sub-MFD-05/sub-MFD-05_ses-ca4ecb4c-4b60-4723-9...,Primary visual area,232,40,1
594,CSH-ZAD-029,e349a2e7-50a3-47ca-bc45-20d1899854ec,sub-CSH-ZAD-029/sub-CSH-ZAD-029_ses-e349a2e7-5...,Primary visual area,542,40,2
3200,NR-0028,5c0c560e-9e1f-45e9-b66e-e4ee7855be84,sub-NR-0028/sub-NR-0028_ses-5c0c560e-9e1f-45e9...,Primary visual area,547,36,2
434,CSH-ZAD-026,5d01d14e-aced-4465-8f8e-9a1c674f62ec,sub-CSH-ZAD-026/sub-CSH-ZAD-026_ses-5d01d14e-a...,Primary visual area,175,34,1


MOs candidates: 30


,subject,session,asset_path,region_parent,n_all_units,n_good_units,n_probes
4733,SWC-038,4d8c7767-981c-4347-8e5e-5d5fffe38534,sub-SWC-038/sub-SWC-038_ses-4d8c7767-981c-4347...,Secondary motor area,1262,135,2
6325,ZM-2240,d33baf74-263c-4b37-a0d0-b79dcb80a764,sub-ZM-2240/sub-ZM-2240_ses-d33baf74-263c-4b37...,Secondary motor area,480,72,1
1077,CSHL058,fb70ebf7-8175-42b0-9b7a-7c6e8612226e,sub-CSHL058/sub-CSHL058_ses-fb70ebf7-8175-42b0...,Secondary motor area,352,70,2
6417,ZM-2241,ff4187b5-4176-4e39-8894-53a24b7cf36b,sub-ZM-2241/sub-ZM-2241_ses-ff4187b5-4176-4e39...,Secondary motor area,440,69,1
3108,NR-0027,0f77ca5d-73c2-45bd-aa4c-4c5ed275dbde,sub-NR-0027/sub-NR-0027_ses-0f77ca5d-73c2-45bd...,Secondary motor area,474,69,2
817,CSHL049,d2918f52-8280-43c0-924b-029b2317e62c,sub-CSHL049/sub-CSHL049_ses-d2918f52-8280-43c0...,Secondary motor area,244,54,1
6100,ZFM-02370,6f6d2c8e-28be-49f4-ae4d-06be2d3148c1,sub-ZFM-02370/sub-ZFM-02370_ses-6f6d2c8e-28be-...,Secondary motor area,203,53,1
2160,KS023,aad23144-0e52-4eac-80c5-c4ee2decb198,sub-KS023/sub-KS023_ses-aad23144-0e52-4eac-80c...,Secondary motor area,166,52,1
287,CSH-ZAD-024,8207abc6-6b23-4762-92b4-82e05bed5143,sub-CSH-ZAD-024/sub-CSH-ZAD-024_ses-8207abc6-6...,Secondary motor area,285,51,1
1194,CSHL060,2e6e179c-fccc-4e8f-9448-ce5b6858a183,sub-CSHL060/sub-CSHL060_ses-2e6e179c-fccc-4e8f...,Secondary motor area,282,50,1
